# 20.5 CI/CD for ML

ML CI/CD gates data, training, evaluation, packaging, and rollout so a model can change quickly without changing blindly.

Versioned artifacts give the pipeline something precise to test; experiment metrics define release thresholds.

Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(205)


def summarize_ci(workload):
    all_gates = np.all(workload["gates"], axis=1)
    bad_release = workload["bad_data"] | workload["bad_metric"] | workload["bad_rollout"]
    escape_rate = float(np.mean(all_gates & bad_release))
    run_count = int(workload["gates"].shape[0])
    seconds = max(float(workload["duration_seconds"]), 1.0)
    return {
        "name": workload["name"],
        "requests": run_count,
        "p50_ms": float(np.percentile(workload["pipeline_latency_ms"], 50)),
        "p95_ms": float(np.percentile(workload["pipeline_latency_ms"], 95)),
        "throughput_qps": run_count / seconds,
        "cost_dollars": float(np.sum(workload["cost_usd"])),
        "drift_stat": float(np.mean(workload["bad_data"])),
        "escape_rate": escape_rate,
    }


def simulate_ci(name, run_count, schema_drift=0.0, metric_flake=0.0, rollback_rate=0.0, software_only=False):
    bad_data = RNG.random(run_count) < schema_drift
    bad_metric = RNG.random(run_count) < metric_flake
    bad_rollout = RNG.random(run_count) < rollback_rate
    schema_gate = ~bad_data
    smoke_gate = RNG.random(run_count) > 0.01
    eval_gate = ~bad_metric
    package_gate = RNG.random(run_count) > 0.005
    rollout_gate = ~bad_rollout
    if software_only:
        schema_gate = np.ones(run_count, dtype=bool)
        eval_gate = np.ones(run_count, dtype=bool)
    gates = np.column_stack([schema_gate, smoke_gate, eval_gate, package_gate, rollout_gate])
    pipeline_latency_ms = RNG.lognormal(mean=np.log(90.0), sigma=0.55, size=run_count)
    cost_usd = 0.004 + pipeline_latency_ms * 0.000002
    return {
        "name": name,
        "gates": gates,
        "bad_data": bad_data,
        "bad_metric": bad_metric,
        "bad_rollout": bad_rollout,
        "pipeline_latency_ms": pipeline_latency_ms,
        "cost_usd": cost_usd,
        "duration_seconds": max(run_count / 4.0, 1.0),
    }


def make_workload_ladder():
    d1 = {
        "name": "D1 five hand Boolean gates",
        "gates": np.array([[True, True, True, True, True], [False, True, True, True, True]]),
        "bad_data": np.array([False, True]),
        "bad_metric": np.array([False, False]),
        "bad_rollout": np.array([False, False]),
        "pipeline_latency_ms": np.array([70.0, 75.0]),
        "cost_usd": np.array([0.004, 0.004]),
        "duration_seconds": 2.0,
    }
    d2 = simulate_ci("D2 1k clean pipeline runs", 1_000)
    d3 = simulate_ci("D3 schema drift and flaky metrics", 10_000, schema_drift=0.06, metric_flake=0.05)
    d4 = simulate_ci("D4 real-ish CI event trace", 80_000, schema_drift=0.03, metric_flake=0.03, rollback_rate=0.02)
    d5 = simulate_ci("D5 production-scale deployment queue", 300_000, schema_drift=0.04, metric_flake=0.04, rollback_rate=0.03)
    return [d1, d2, d3, d4, d5]


## The concept, built once: ML release gates

The lesson formula is $\text{release}=1$ only when every gate $g_k=1$. We implement the gate product and assert the schema mismatch $24-23=1$.

In [ ]:

def ml_release_gates(schema, smoke, eval_gate, package, rollout):
    gates = np.array([schema, smoke, eval_gate, package, rollout], dtype=bool)
    return bool(np.all(gates)), gates

release, gates = ml_release_gates(True, True, True, True, True)
blocking_mismatch = 24 - 23
assert release is True
assert blocking_mismatch == 1
print("release", release)
print("blocking schema mismatch", blocking_mismatch)


The other lesson checks are cheap but operational: smoke loss drops $0.90-0.62=0.28$, evaluation clears by $0.190-0.184=0.006$, and a canary with $2000\times0.004=8$ errors is countable.

In [ ]:

loss_drop = 0.90 - 0.62
eval_margin = 0.190 - 0.184
canary_errors = 2000 * 0.004
latency_over_budget = 180 - 150
assert np.isclose(loss_drop, 0.28)
assert np.isclose(eval_margin, 0.006)
assert np.isclose(canary_errors, 8.0)
assert latency_over_budget == 30
print("loss drop", loss_drop)
print("eval margin", eval_margin)
print("canary errors", canary_errors)


## The dataset ladder

Family F17 has no shared ML-training ladder here. We build the operational workload ladder inline: D1 tiny trace, D2 larger volume, D3 spikes or drift, D4 real-ish synthetic trace, and D5 production-scale simulation.

In [ ]:

workloads = make_workload_ladder()
for workload in workloads:
    print(workload["name"])
    print("shape", workload["gates"].shape)
    print("first gate rows", workload["gates"][:3])


## Run the same method across D1-D5

The metric is failed-release escape rate: all gates pass even though data, metric, or rollout state is bad. The same summary reports latency, cost, throughput, and data-drift rate.

In [ ]:

workloads = make_workload_ladder()
summaries = [summarize_ci(workload) for workload in workloads]

header = f"{'rung':<42} {'load':>10} {'p50':>9} {'p95':>9} {'cost':>10} {'qps':>10} {'escape_rate':>12}"
print(header)
for row in summaries:
    line = f"{row['name']:<42} {row['requests']:>10} {row['p50_ms']:>9.2f} {row['p95_ms']:>9.2f} {row['cost_dollars']:>10.3f} {row['throughput_qps']:>10.2f} {row['escape_rate']:>12.5f}"
    print(line)


The first row is the operational artifact: p95 latency, total cost, and throughput by rung. The second plot is the lesson metric versus load.

In [ ]:

names = [row["name"].split()[0] for row in summaries]
loads = np.array([row["requests"] for row in summaries], dtype=float)
p95_values = np.array([row["p95_ms"] for row in summaries], dtype=float)
cost_values = np.array([row["cost_dollars"] for row in summaries], dtype=float)
throughput_values = np.array([row["throughput_qps"] for row in summaries], dtype=float)
metric_values = np.array([row["escape_rate"] for row in summaries], dtype=float)
cost_per_request = cost_values / loads
normalized_p95 = p95_values / max(float(np.max(p95_values)), 1.0)
normalized_cost = cost_per_request / max(float(np.max(cost_per_request)), 1.0)
normalized_throughput = throughput_values / max(float(np.max(throughput_values)), 1.0)

fig = plt.figure(figsize=(15, 7))
grid = fig.add_gridspec(2, 5, height_ratios=[1.0, 1.15])
for index, name in enumerate(names):
    ax = fig.add_subplot(grid[0, index])
    values = [normalized_p95[index], normalized_cost[index], normalized_throughput[index]]
    ax.bar(["p95", "cost", "qps"], values)
    ax.set_ylim(0.0, 1.05)
    ax.set_title(name)
    ax.tick_params(axis="x", rotation=30)
    ax.set_ylabel("normalized")
summary_ax = fig.add_subplot(grid[1, :])
summary_ax.plot(loads, metric_values, marker="o")
summary_ax.set_xscale("log")
summary_ax.set_title("failed-release escape rate vs load")
summary_ax.set_xlabel("records or requests")
summary_ax.set_ylabel("failed-release escape rate")
fig.suptitle("Per-workload latency, cost, throughput, and metric-vs-load")
fig.tight_layout()


## Pitfall on D5: only testing code

Software-only gates ignore schema and metric failures. The fix is to include schema and metric gates in the release product so bad data cannot pass as good software.

In [ ]:

d5_software_only = simulate_ci("D5 software-only gates", 300_000, schema_drift=0.04, metric_flake=0.04, rollback_rate=0.03, software_only=True)
d5_full = workloads[-1]
software_only_escape = summarize_ci(d5_software_only)["escape_rate"]
full_gate_escape = summarize_ci(d5_full)["escape_rate"]
print("software-only escape rate", round(software_only_escape, 5))
print("full-gate escape rate", round(full_gate_escape, 5))


## Evaluate it + Practice

- Metric: failed-release escape rate; no-skill baseline tests packaging only.
- Sanity check: if any D1 gate is false, release must be false.
- Ablation: remove schema and metric gates and watch bad data escape.
- Failure signals: flaky metrics, repeated rollbacks, or live p95 over budget without automatic rollback.

Practice: turn the package gate off and confirm the release product is zero.

Practice: change the maximum allowed error from $0.190$ to $0.180$ and recompute the eval gate.

Practice: compute canary errors for 5,000 requests at 0.6 percent error.